RAG with PDF as context source
Install the dependencies first
pip install pypdf


In [1]:
#importing libraries
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. SETUP
# ─────────────────────────────────────────

print("🔧 Loading models...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
llm = OllamaLLM(model="llama3.2:3b")
client = chromadb.Client()
collection = client.create_collection("attention_paper")
print("✅ Models ready\n")

🔧 Loading models...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1160.26it/s]


✅ Models ready



In [3]:
# 2. LOAD PDF
# ─────────────────────────────────────────

loader = PyPDFLoader("NIPS-2017-attention-is-all-you-need-Paper.pdf")  # ← your filename here
pages = loader.load()
print(f"✅ Loaded {len(pages)} pages")
print(f"📄 Sample from page 1:\n{pages[0].page_content[:300]}\n")

✅ Loaded 11 pages
📄 Sample from page 1:
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aid



In [4]:
# 3. CHUNK
# ─────────────────────────────────────────

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(pages)
texts = [chunk.page_content for chunk in chunks]
print(f"✅ Split into {len(texts)} chunks")
print(f"📦 Sample chunk:\n{texts[0]}\n")


✅ Split into 76 chunks
📦 Sample chunk:
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or



In [5]:
# 4. EMBED + STORE IN CHROMADB
# ─────────────────────────────────────────

print("⏳ Embedding chunks...")
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=[f"chunk{i}" for i in range(len(texts))]
)
print(f"\n✅ Indexed {len(texts)} chunks into ChromaDB\n")

⏳ Embedding chunks...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.12it/s]


✅ Indexed 76 chunks into ChromaDB



In [6]:
# 5. RAG FUNCTION
# ─────────────────────────────────────────

def ask(question):
    print(f"❓ Question: {question}\n")

    # Embed the question
    query_embedding = embedding_model.encode(question).tolist()

    # Retrieve top 4 relevant chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=4
    )
    retrieved_chunks = results["documents"][0]

    # Build prompt
    context = "\n\n".join(retrieved_chunks)
    prompt = f"""You are an expert AI research assistant.
Answer the question using ONLY the context provided below.
Be specific and reference details from the context.
If the answer is not in the context, say "The paper doesn't cover that."

Context:
{context}

Question: {question}

Answer:"""

    print(f"📋 Retrieved {len(retrieved_chunks)} chunks")
    print(f"🤖 Answer:")
    response = llm.invoke(prompt)
    print(response)
    print("\n" + "─"*60 + "\n")

In [7]:
# 6. ASK QUESTIONS ABOUT THE PAPER
# ─────────────────────────────────────────
ask("Who are the authors of this paper?")

❓ Question: Who are the authors of this paper?

📋 Retrieved 4 chunks
🤖 Answer:
The authors of the papers listed in the context are not explicitly stated as being the same. However, Rafal Jozefowicz, Oriol Vinyals, Mike Schuster, Noam Shazeer, and Yonghui Wu co-authored "Exploring the limits of language modeling" ([13]).

────────────────────────────────────────────────────────────



In [8]:
ask("What is this paper about?")

❓ Question: What is this paper about?

📋 Retrieved 4 chunks
🤖 Answer:
This paper appears to be about the Transformer model, specifically its architecture and components such as attention and masking mechanisms. It does not provide explicit information on a specific topic or problem that the paper is addressing, but based on the context of being part of the research at Google Brain and Google Research, it can be inferred that the paper may focus on improving or exploring the Transformer model for tasks like neural machine translation.

────────────────────────────────────────────────────────────



In [9]:
ask("What BLEU scores did the model achieve on translation tasks?")

❓ Question: What BLEU scores did the model achieve on translation tasks?

📋 Retrieved 4 chunks
🤖 Answer:
According to the context, the model achieved the following BLEU scores:

* On the WMT 2014 English-to-German translation task, the big Transformer model achieved a new state-of-the-art BLEU score of 28.4.
* On the WMT 2014 English-to-French translation task, the same big Transformer model achieved a BLEU score of 41.0.

────────────────────────────────────────────────────────────



In [ ]:
ask("how many moons does Jupiter have?")
#out of scope question to test if the model is just making up answers or admitting when it doesn't know something

❓ Question: how many moons does Jupiter have?

📋 Retrieved 4 chunks
🤖 Answer:
The paper doesn't cover that. The provided context is about AI research, specifically transformer models and their training costs, and does not mention astronomy or the number of moons on Jupiter.

────────────────────────────────────────────────────────────

